# Phase 3: Xử lý Mất Cân Bằng Dữ Liệu (Class Imbalance Handling)
## CICIoT2023 Network Intrusion Detection
### Đề tài Tốt nghiệp: Hệ thống phát hiện tấn công mạng IoT dựa trên XAI

---

### 🎯 Mục tiêu Phase 3
1. **Phân tích định lượng độ mất cân bằng**: Tính toán Imbalance Ratio (IR), Shannon Entropy, và phân tầng mức độ nghiêm trọng (Majority, Medium, Minority, Extreme).
2. **Phương pháp Paper (SMOTE + Tomek Links - `SMOTETomek`)**:
   - **SMOTE (Synthetic Minority Over-sampling Technique)**: Tạo mẫu tổng hợp nội suy cho các lớp tấn công thiểu số thay vì sao chép ngẫu nhiên (tránh overfitting).
   - **Tomek Links**: Phát hiện và loại bỏ các cặp điểm lân cận gần nhất khác nhãn (nhiễu biên / ambiguous borderline samples), làm sắc nét ranh giới phân lớp.
3. **Kiến trúc Hybrid Scalable cho Dữ liệu Lớn (Big Data IoT)**:
   - Kết hợp **Smart Undersampling** lớp đa số khổng lồ (DDoS, DoS) + **SMOTE** lớp thiểu số (Web, BruteForce, Malware) + **Tomek Links** làm sạch ranh giới.
4. **Học nhạy cảm chi phí (Cost-Sensitive Learning - Class Weights)**:
   - Tính toán trọng số nghịch đảo tần suất (`compute_class_weight`) cho cả 3 cấp độ: 34 Attacks, 8 Categories, Binary.
   - Giải pháp $O(1)$ về bộ nhớ RAM, bảo toàn phân bố tự nhiên của lưu lượng mạng thực tế.
5. **Trực quan hóa không gian quyết định (Decision Space)**:
   - Biểu đồ PCA 2D minh họa cơ chế xóa nhiễu biên của Tomek Links.

In [ ]:
# ==============================================================================
# Cell 1: Khởi tạo Môi trường & Nạp Thư viện Cần thiết
# ==============================================================================
import os
import gc
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.under_sampling import TomekLinks, RandomUnderSampler
from imblearn.combine import SMOTETomek, SMOTEENN

warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

os.makedirs('plots', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Nạp dữ liệu từ Phase 2
print("Đang nạp dữ liệu từ Phase 2...")
for p in ['X_train.pkl', 'data/X_train.pkl']:
    if os.path.exists(p):
        X_train = joblib.load(p)
        print(f"  Đã nạp X_train: {X_train.shape[0]:,} dòng x {X_train.shape[1]} đặc trưng")
        break

for p in ['y_train.pkl', 'data/y_train.pkl']:
    if os.path.exists(p):
        y_train_dict = joblib.load(p)
        print(f"  Đã nạp y_train keys: {list(y_train_dict.keys())}")
        break

preprocessor = joblib.load('models/preprocessor.joblib')
label_mapping = preprocessor['label_mapping']
category_mapping = preprocessor['category_mapping']
reverse_label_map = {v: k for k, v in label_mapping.items()}
reverse_cat_map = {v: k for k, v in category_mapping.items()}
print("Sẵn sàng xử lý mất cân bằng dữ liệu!")

## Task 3.1: Phân tích Định lượng Độ Mất Cân Bằng (Imbalance Analysis)

Đo lường mức độ lệch phân bố qua 2 chỉ số học thuật chính:
1. **Imbalance Ratio (IR)**: Tỷ số giữa số lượng mẫu lớp đa số lớn nhất và lớp thiểu số nhỏ nhất:
   $$IR = \frac{\max_i (N_i)}{\min_i (N_i)}$$
2. **Shannon Entropy Chuẩn hóa**: Đo lường độ đồng đều thông tin của phân bố xác suất các lớp:
   $$H(Y) = -\sum_{i=1}^{C} p_i \log_2(p_i), \quad \text{Entropy Ratio} = \frac{H(Y)}{\log_2(C)} \in [0, 1]$$
   - Giá trị càng gần 1 $\rightarrow$ phân bố càng cân bằng.
   - Giá trị càng gần 0 $\rightarrow$ phân bố cực kỳ lệch về một vài lớp chiếm ưu thế.

In [ ]:
# ==============================================================================
# Cell 2: Tính toán Chi tiết Chỉ số Mất Cân Bằng
# ==============================================================================
y_train_encoded = y_train_dict['label_encoded']
y_train_names = y_train_dict['label_name']
y_cat_encoded = y_train_dict['category_encoded']
y_cat_names = y_train_dict['category_name']
y_binary = y_train_dict['is_attack']

total_samples = len(y_train_names)
label_counts = pd.Series(y_train_names).value_counts()
label_pct = (label_counts / total_samples) * 100

cat_counts = pd.Series(y_cat_names).value_counts()
cat_pct = (cat_counts / total_samples) * 100

# IR
ir_fine = label_counts.max() / label_counts.min()
ir_cat = cat_counts.max() / cat_counts.min()

# Shannon Entropy
p_fine = label_counts.values / total_samples
ent_fine = -np.sum(p_fine * np.log2(p_fine + 1e-12)) / np.log2(len(label_counts))

p_cat = cat_counts.values / total_samples
ent_cat = -np.sum(p_cat * np.log2(p_cat + 1e-12)) / np.log2(len(cat_counts))

# Phân tầng 4 mức độ
majority = label_pct[label_pct >= 5.0].index.tolist()
medium = label_pct[(label_pct >= 1.0) & (label_pct < 5.0)].index.tolist()
minority = label_pct[(label_pct >= 0.1) & (label_pct < 1.0)].index.tolist()
extreme = label_pct[label_pct < 0.1].index.tolist()

print(f"Tổng số mẫu Train:       {total_samples:,} flows")
print(f"Số lớp chi tiết:         {len(label_counts)} | Số nhóm (Categories): {len(cat_counts)}")
print(f"Imbalance Ratio (34):    {ir_fine:,.1f}:1 ({label_counts.idxmax()}={label_counts.max():,} vs {label_counts.idxmin()}={label_counts.min():,})")
print(f"Imbalance Ratio (Nhóm):  {ir_cat:,.1f}:1 ({cat_counts.idxmax()}={cat_counts.max():,} vs {cat_counts.idxmin()}={cat_counts.min():,})")
print(f"Shannon Entropy Ratio:   {ent_fine:.4f} (34 lớp) | {ent_cat:.4f} (Nhóm)")
print("-" * 70)
print(f"[1] Majority (>= 5%):     {len(majority):2d} lớp ({sum(label_pct[c] for c in majority):.2f}% lưu lượng)")
print(f"[2] Medium (1% - 5%):     {len(medium):2d} lớp ({sum(label_pct[c] for c in medium):.2f}% lưu lượng)")
print(f"[3] Minority (0.1% - 1%): {len(minority):2d} lớp ({sum(label_pct[c] for c in minority):.2f}% lưu lượng)")
print(f"[4] Extreme (< 0.1%):     {len(extreme):2d} lớp ({sum(label_pct[c] for c in extreme):.2f}% lưu lượng)")

## Trực quan hóa Mức độ Mất Cân Bằng (Plot 06)
Biểu đồ phân bố 34 lớp tấn công trên thang logarit và biểu đồ cột thể hiện số lượng lớp theo từng mức độ nghiêm trọng.

In [ ]:
# ==============================================================================
# Cell 3: Vẽ Biểu đồ Phân tích Mất Cân Bằng
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# Trục 1: Phân bố 34 lớp log scale
ax1 = axes[0]
colors_bar = plt.cm.plasma(np.linspace(0.15, 0.85, len(label_counts)))[::-1]
ax1.barh(range(len(label_counts)), label_counts.values, color=colors_bar, edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(len(label_counts)))
ax1.set_yticklabels(label_counts.index, fontsize=8)
ax1.set_xscale('log')
ax1.set_xlabel('Số lượng mẫu (Thang Logarit)', fontsize=11, fontweight='bold')
ax1.set_title(f'Phân bố 34 Lớp Tấn công CICIoT2023\nImbalance Ratio = {ir_fine:,.1f}:1', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5, axis='x')
ax1.invert_yaxis()

# Trục 2: Phân tầng mức độ nghiêm trọng
ax2 = axes[1]
tier_labels = ['Majority\n(>=5%)', 'Medium\n(1%-5%)', 'Minority\n(0.1%-1%)', 'Extreme\n(<0.1%)']
tier_counts = [len(majority), len(medium), len(minority), len(extreme)]
tier_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
bars = ax2.bar(tier_labels, tier_counts, color=tier_colors, edgecolor='black', linewidth=0.8, width=0.55)
ax2.set_ylabel('Số lượng lớp tấn công', fontsize=11, fontweight='bold')
ax2.set_title('Phân loại Mức độ Mất Cân Bằng (Severity Breakdown)', fontsize=12, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.5, axis='y')
for bar, count in zip(bars, tier_counts):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f"{count} lớp", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/06_imbalance_analysis.png', dpi=200, bbox_inches='tight')
plt.show()

## Task 3.2A: Chiến lược Học Nhạy Cảm Chi Phí (Cost-Sensitive Learning)

### Công thức Tính Trọng số Lớp (Inverse-Frequency Class Weights)
Trọng số của mỗi lớp $c$ được tính toán theo công thức chuẩn Scikit-learn:
$$w_c = \frac{N}{C \times N_c}$$
Trong đó:
- $N$: Tổng số mẫu trong tập huấn luyện ($~5.49$ triệu flows).
- $C$: Tổng số lớp ($34$ lớp hoặc $8$ nhóm hoặc $2$ nhị phân).
- $N_c$: Số lượng mẫu của lớp $c$.

**Ưu điểm vượt trội với CICIoT2023**:
- Chi phí tính toán cực nhỏ ($O(1)$ RAM), không làm méo mó phân bố thực tế của mạng.
- Khi tính hàm mất mát (Log Loss / Cross-Entropy), các mẫu của lớp cực hiếm như `Uploading_Attack`, `Malware` sẽ nhân với trọng số phạt lớn gấp hàng trăm lần, buộc Gradient Descent phải cập nhật trọng số cho chúng.

In [ ]:
# ==============================================================================
# Cell 4: Tính toán Class Weights cho Cả 3 Cấp độ Phân loại
# ==============================================================================
# 1. 34 Lớp chi tiết
unique_labels = np.unique(y_train_encoded)
cw_arr = compute_class_weight('balanced', classes=unique_labels, y=y_train_encoded)
cw_encoded = dict(zip(unique_labels.tolist(), cw_arr.tolist()))
cw_named = {reverse_label_map[c]: round(w, 6) for c, w in cw_encoded.items()}

# 2. Nhóm tấn công (Categories)
unique_cats = np.unique(y_cat_encoded)
cat_arr = compute_class_weight('balanced', classes=unique_cats, y=y_cat_encoded)
cat_encoded = dict(zip(unique_cats.tolist(), cat_arr.tolist()))
cat_named = {reverse_cat_map[c]: round(w, 6) for c, w in cat_encoded.items()}

# 3. Nhị phân (Benign vs Attack)
unique_bin = np.unique(y_binary)
bin_arr = compute_class_weight('balanced', classes=unique_bin, y=y_binary)
bin_weights = dict(zip(unique_bin.tolist(), bin_arr.tolist()))

print(f"Trọng số 34 lớp chi tiết: Max = {max(cw_named.items(), key=lambda x: x[1])}, Min = {min(cw_named.items(), key=lambda x: x[1])}")
print("\nTrọng số các nhóm tấn công (Categories):")
for k, v in sorted(cat_named.items(), key=lambda x: x[1], reverse=True):
    print(f"  {k:<15}: {v:8.4f}")
print(f"\nTrọng số nhị phân: Benign (0) = {bin_weights[0]:.4f} | Attack (1) = {bin_weights[1]:.4f}")

## Trực quan hóa Class Weights (Plot 07)
Biểu đồ phân bố trọng số của 34 lớp và 8 nhóm tấn công.

In [ ]:
# ==============================================================================
# Cell 5: Vẽ Biểu đồ Class Weights
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Trục 1: Fine-grained class weights (Log Scale)
ax1 = axes[0]
sorted_cw = sorted(cw_named.items(), key=lambda x: x[1])
names_w = [x[0] for x in sorted_cw]
vals_w = [x[1] for x in sorted_cw]
colors_w = plt.cm.viridis(np.linspace(0.1, 0.95, len(vals_w)))
ax1.barh(range(len(names_w)), vals_w, color=colors_w, edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(len(names_w)))
ax1.set_yticklabels(names_w, fontsize=7.5)
ax1.set_xscale('log')
ax1.set_xlabel('Giá trị Class Weight (Thang Logarit)', fontsize=11, fontweight='bold')
ax1.set_title('Trọng số Class Weights Nghịch đảo (34 Lớp Tấn công)', fontsize=12, fontweight='bold')
ax1.axvline(x=1.0, color='red', linestyle='--', linewidth=1.2, label='Neutral Weight = 1.0')
ax1.grid(True, linestyle='--', alpha=0.5, axis='x')
ax1.legend()

# Trục 2: Category weights
ax2 = axes[1]
sorted_cat = sorted(cat_named.items(), key=lambda x: x[1])
names_cat = [x[0] for x in sorted_cat]
vals_cat = [x[1] for x in sorted_cat]
bars_cat = ax2.bar(names_cat, vals_cat, color='#3498db', edgecolor='black', linewidth=0.8, width=0.55)
ax2.set_ylabel('Giá trị Trọng số', fontsize=11, fontweight='bold')
ax2.set_title('Trọng số Nghịch đảo cho 8 Nhóm Tấn công + Benign', fontsize=12, fontweight='bold')
ax2.set_xticklabels(names_cat, rotation=35, ha='right', fontsize=10, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.5, axis='y')
for bar, val in zip(bars_cat, vals_cat):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
             f"{val:.2f}", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/07_class_weights.png', dpi=200, bbox_inches='tight')
plt.show()

## Task 3.2B: Phương pháp Resampling — SMOTE + Tomek Links (`SMOTETomek`)

### 1. Cơ sở Lý thuyết của Paper
Trong bài báo nghiên cứu, kỹ thuật kết hợp **SMOTE + Tomek Links** được chọn vì:
- **SMOTE**: Không sao chép mẫu thô sơ (như RandomOverSampler) mà tạo mẫu nội suy nhân tạo $\mathbf{x}_{\text{new}} = \mathbf{x}_i + \lambda (\mathbf{x}_{zi} - \mathbf{x}_i)$ với $\lambda \sim U(0,1)$ dựa trên $k$ láng giềng gần nhất ($k$-NN).
- **Tomek Links**: Khi SMOTE tạo mẫu, có khả năng mẫu nhân tạo rơi vào vùng ranh giới giữa lớp thiểu số và đa số gây chồng lấn (overlapping / borderline noise). Tomek Links xác định mọi cặp $(\mathbf{x}_i, \mathbf{x}_j)$ khác nhãn mà khoảng cách giữa chúng ngắn hơn tới bất kỳ điểm nào khác, sau đó loại bỏ điểm gây nhiễu để tạo ranh giới phân tách sạch (clean decision boundary).

### 2. Thách thức trên Tập dữ liệu Lớn (Big Data IoT ~5.5 Triệu dòng)
Thuật toán $k$-NN trong Tomek Links có độ phức tạp tính toán lớn nếu chạy trực tiếp trên toàn bộ 5.5 triệu dòng $\times$ 58 chiều float32. Vì vậy, ta áp dụng kiến trúc **Two-Stage Scalable Hybrid**:
1. **Stage 1 (Pruning Majority)**: Tỉa bớt các lớp đa số khổng lồ (DDoS, DoS chiếm ~90%) xuống ngưỡng an toàn (`TARGET_MAJORITY`).
2. **Stage 2 (SMOTE Synthesis)**: Nội suy tạo mẫu nhân tạo cho các lớp thiểu số cực hiếm (Web, BruteForce, Malware, Benign) lên mức `TARGET_MINORITY`.
3. **Stage 3 (Tomek Links Boundary Cleaning)**: Chạy Tomek Links song song đa nhân (`n_jobs=-1`) trên tập kết hợp để xóa sạch các điểm nhiễu biên.

In [ ]:
# ==============================================================================
# Cell 6: Thực thi Two-Stage Scalable Resampling (SMOTE + Tomek Links)
# ==============================================================================
SAMPLE_SIZE = 100000       # Lấy mẫu phân tầng đại diện 100,000 dòng
TARGET_MAJORITY = 30000    # Pruning lớp đa số xuống tối đa 30,000 dòng
TARGET_MINORITY = 8000     # SMOTE lớp thiểu số lên 8,000 dòng
K_NEIGHBORS = 5
RANDOM_STATE = 42

print("=" * 75)
print("TASK 3.2B: THỰC THI PIPELINE RESAMPLING (SMOTE + TOMEK LINKS)")
print("=" * 75)

# 1. Lấy mẫu đại diện phân tầng (Stratified Sampling)
_, X_sub, _, y_sub = train_test_split(
    X_train, y_cat_encoded,
    test_size=SAMPLE_SIZE,
    stratify=y_cat_encoded,
    random_state=RANDOM_STATE
)
X_sub = pd.DataFrame(X_sub, columns=X_train.columns) if hasattr(X_train, 'columns') else pd.DataFrame(X_sub)
y_sub = np.array(y_sub)

counts_init = pd.Series(y_sub).value_counts()
print(f"Tập mẫu ban đầu: {len(y_sub):,} dòng across {len(counts_init)} nhóm")

# 2. Stage 1: Smart Majority Pruning
strat_under = {c: TARGET_MAJORITY for c, cnt in counts_init.items() if cnt > TARGET_MAJORITY}
if strat_under:
    print(f"  [Stage 1: Undersampling] Tỉa {len(strat_under)} lớp đa số xuống {TARGET_MAJORITY:,}...")
    rus = RandomUnderSampler(sampling_strategy=strat_under, random_state=RANDOM_STATE)
    X_work, y_work = rus.fit_resample(X_sub, y_sub)
else:
    X_work, y_work = X_sub.copy(), y_sub.copy()

# 3. Stage 2: SMOTE Synthesis
counts_mid = pd.Series(y_work).value_counts()
strat_over = {c: TARGET_MINORITY for c, cnt in counts_mid.items() if cnt < TARGET_MINORITY}
if strat_over:
    min_k = min(counts_mid[c] for c in strat_over.keys())
    eff_k = max(1, min(K_NEIGHBORS, min_k - 1))
    print(f"  [Stage 2: SMOTE Synthesis] Nội suy {len(strat_over)} lớp thiểu số lên {TARGET_MINORITY:,} (k={eff_k})...")
    smote = SMOTE(sampling_strategy=strat_over, k_neighbors=eff_k, random_state=RANDOM_STATE)
    X_work, y_work = smote.fit_resample(X_work, y_work)

# 4. Stage 3: Tomek Links Boundary Cleaning
print("  [Stage 3: Tomek Links] Đang phát hiện và loại bỏ các cặp điểm nhiễu biên (Borderline Pairs)...")
t0 = time.time()
tomek = TomekLinks(sampling_strategy='all', n_jobs=-1)
X_clean, y_clean = tomek.fit_resample(X_work, y_work)
elapsed_tomek = time.time() - t0
print(f"  -> Hoàn tất Tomek Links trong {elapsed_tomek:.2f} giây!")

counts_final = pd.Series(y_clean).value_counts()
print("-" * 75)
print(f"Số dòng biến thiên: {len(y_sub):,} -> {len(y_clean):,} dòng")
print(f"Imbalance Ratio:    {counts_init.max()/counts_init.min():,.1f}:1  ->  {counts_final.max()/counts_final.min():.2f}:1")

## Trực quan hóa So sánh Trước và Sau Resampling (Plot 08 & 08b)
- **Plot 08**: So sánh số lượng mẫu từng nhóm trước và sau khi áp dụng SMOTE + Tomek Links.
- **Plot 08b**: Trực quan hóa không gian quyết định qua **PCA 2D**, kiểm chứng hiệu quả phân tách ranh giới rõ ràng sau khi loại bỏ nhiễu biên.

In [ ]:
# ==============================================================================
# Cell 7: Vẽ Biểu đồ So sánh Phân bố Trước vs Sau Resampling
# ==============================================================================
fig, ax = plt.subplots(figsize=(15, 7))
common_indices = counts_init.index
x_indices = np.arange(len(common_indices))
bar_width = 0.38

counts_b = [counts_init.get(k, 0) for k in common_indices]
counts_a = [counts_final.get(k, 0) for k in common_indices]
tick_labels = [reverse_cat_map.get(k, str(k)) for k in common_indices]

ax.bar(x_indices - bar_width/2, counts_b, width=bar_width,
       label='Ban đầu (Gốc)', color='#e74c3c', alpha=0.85, edgecolor='black')
ax.bar(x_indices + bar_width/2, counts_a, width=bar_width,
       label='Sau Resampling (SMOTE + Tomek Links)', color='#2ecc71', alpha=0.85, edgecolor='black')

ax.set_ylabel('Số lượng mẫu (Thang Logarit)', fontsize=11, fontweight='bold')
ax.set_yscale('log')
ax.set_title('So sánh Phân bố Số lượng Mẫu Trước và Sau khi Cân bằng (SMOTE + Tomek Links)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x_indices)
ax.set_xticklabels(tick_labels, rotation=30, ha='right', fontsize=10, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5, axis='y')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('plots/08_resampling_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# Cell 8: Trực quan hóa Không gian Quyết định 2D PCA sau Tomek Links
# ==============================================================================
print("Đang giảm chiều PCA để trực quan hóa 2D không gian quyết định...")
n_vis = min(5000, len(y_clean))
idx_vis = np.random.choice(len(y_clean), size=n_vis, replace=False)
X_vis = X_clean.iloc[idx_vis] if hasattr(X_clean, 'iloc') else X_clean[idx_vis]
y_vis = y_clean[idx_vis]

pca = PCA(n_components=2, random_state=42)
X_pca_2d = pca.fit_transform(X_vis)

fig, ax = plt.subplots(figsize=(12, 8))
unique_cls = np.unique(y_vis)
palette = plt.cm.tab10(np.linspace(0, 1, len(unique_cls)))

for c, color in zip(unique_cls, palette):
    mask = (y_vis == c)
    name = reverse_cat_map.get(c, f'Class {c}')
    ax.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
               c=[color], label=name, alpha=0.6, s=18, edgecolor='none')

ax.set_title(f'Không gian Quyết định 2D (PCA) sau khi Tomek Links Làm sạch Nhiễu biên\nTổng phương sai giải thích: {pca.explained_variance_ratio_.sum()*100:.1f}%',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Principal Component 1 (Thành phần chính 1)', fontweight='bold')
ax.set_ylabel('Principal Component 2 (Thành phần chính 2)', fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9.5)

plt.tight_layout()
plt.savefig('plots/08b_smote_tomek_boundary.png', dpi=200, bbox_inches='tight')
plt.show()

## Task 3.3: Lưu trữ Toàn bộ Kết quả Tiền Xử lý Phase 3
Lưu trữ các deliverables chuẩn hóa để sẵn sàng cho các phase tiếp theo:
1. `class_weights.pkl` & `data/class_weights.pkl`: Bộ trọng số lớp cho 3 cấp độ.
2. `models/class_weights.json`: Định dạng JSON dễ đọc.
3. `data/X_train_balanced.pkl` & `data/y_train_balanced.pkl`: Tập dữ liệu đã cân bằng.
4. `models/preprocessor.joblib`: Cập nhật pipeline với class weights.
5. `models/resampling_summary.json`: Báo cáo chi tiết tham số và số lượng mẫu.

In [ ]:
# ==============================================================================
# Cell 9: Lưu trữ Deliverables
# ==============================================================================
all_weights = {
    'fine_grained_encoded': cw_encoded,
    'fine_grained_named': cw_named,
    'category_encoded': cat_encoded,
    'category_named': cat_named,
    'binary': bin_weights
}

# 1. Save PKL
joblib.dump(all_weights, 'class_weights.pkl')
joblib.dump(all_weights, 'data/class_weights.pkl')
print("Đã lưu: class_weights.pkl & data/class_weights.pkl")

# 2. Save JSON
weights_json = {
    'fine_grained': cw_named,
    'categories': cat_named,
    'binary': {str(k): v for k, v in bin_weights.items()}
}
with open('models/class_weights.json', 'w') as f:
    json.dump(weights_json, f, indent=4)
print("Đã lưu: models/class_weights.json")

# 3. Save Balanced Dataset
joblib.dump(X_clean, 'data/X_train_balanced.pkl', compress=3)
joblib.dump(y_clean, 'data/y_train_balanced.pkl', compress=3)
print("Đã lưu: data/X_train_balanced.pkl, data/y_train_balanced.pkl")

# 4. Update preprocessor
preprocessor['class_weights_encoded'] = cw_encoded
preprocessor['class_weights_named'] = cw_named
preprocessor['category_weights_encoded'] = cat_encoded
preprocessor['binary_weights'] = bin_weights
joblib.dump(preprocessor, 'models/preprocessor.joblib')
print("Đã cập nhật: models/preprocessor.joblib")

# 5. Save Resampling Summary
res_summary = {
    'strategy': 'Hybrid Scalable SMOTE + Tomek Links (Paper-Aligned)',
    'initial_sample_size': len(y_sub),
    'final_sample_size': len(y_clean),
    'initial_ir': round(counts_init.max() / counts_init.min(), 2),
    'final_ir': round(counts_final.max() / counts_final.min(), 2),
    'target_majority': TARGET_MAJORITY,
    'target_minority': TARGET_MINORITY,
    'k_neighbors': K_NEIGHBORS
}
with open('models/resampling_summary.json', 'w') as f:
    json.dump(res_summary, f, indent=4)
print("Đã lưu: models/resampling_summary.json")
print("\nPhase 3 hoàn thành xuất sắc!")